In [1]:
import pandas as pd
df = pd.read_csv("../data/processed/labeled_part_1 - labeled_part_1.csv")
df

,review,labels
0,Dont buy the business combo or flexibility or ...,pricing_fees_negative
1,I wasnt quite sure what to expect having read ...,checkin_boarding_process_positive | inflight_e...
2,Los Angeles to Copenhagen via Reykjavik. Ive m...,flight_delay_cancellation_negative | online_bo...
3,Never had such a poor experience time after ti...,checkin_boarding_process_negative | pricing_fe...
4,Fabulous flights with AirAsia X yet again. Bot...,flight_delay_cancellation_negative | inflight_...
...,...,...
4995,Southwest has had major delays on my past four...,flight_delay_cancellation_negative
4996,For a 3.45 hour Helsinki to Athens flight we g...,flight_delay_cancellation_negative | inflight_...
4997,My girlfriend and I flew LIAT from into St. Ma...,flight_delay_cancellation_negative | checkin_b...
4998,Bad business practices on multiple levels. I h...,pricing_fees_negative


In [49]:
import pandas as pd
import numpy as np

def parse_multilabel_data(df):
    """
    Parse multi-label sentiment data from CSV.
    
    Args:
        input_file: Path to input CSV file (index, review_text, labels)
        output_file: Optional path to save output CSV
    
    Returns:
        DataFrame with one-hot encoded labels
    """
    tags = [
        "flight_delay_cancellation",
        "checkin_boarding_process",
        "baggage_issues",
        "inflight_experience",
        "pricing_fees",
        "online_booking",
    ]

    # Define all possible tags
    all_tags = [
        "flight_delay_cancellation_negative",
        "flight_delay_cancellation_positive",
        "checkin_boarding_process_negative",
        "checkin_boarding_process_positive",
        "baggage_issues_negative",
        "baggage_issues_positive",
        "inflight_experience_negative",
        "inflight_experience_positive",
        "pricing_fees_negative",
        "pricing_fees_positive",
        "online_booking_negative",
        "online_booking_positive"
    ]
    results = []
    for ind, row in df.iterrows():
        result = {tag: 0 for tag in tags}
        
        label_string = row['labels']  # Get the label string from the row
        
        if pd.isna(label_string) or label_string == "":
            continue  # Skip to next iteration instead of returning
        
        labels = [x.strip() for x in label_string.split("|")]
        
        for label in labels:
            # Find which base tag this belongs to
            for tag in tags:
                if label.startswith(tag):
                    if label.endswith("_negative"):
                        result[tag] = -1
                    elif label.endswith("_positive"):
                        result[tag] = 1
                    break
        results.append(result)  # ADD THIS LINE - append to results list
        print(result)  # Print the result for this row

    print(results)
    df_encoded = pd.DataFrame(results)
    return(df_encoded)

# Process all rows
df_encoded = parse_multilabel_data(df)

# Merge with original dataframe
df_merged = pd.concat([df.reset_index(drop=True), df_encoded.reset_index(drop=True)], axis=1)
  

{'flight_delay_cancellation': 0, 'checkin_boarding_process': 0, 'baggage_issues': 0, 'inflight_experience': 0, 'pricing_fees': -1, 'online_booking': 0}
{'flight_delay_cancellation': 0, 'checkin_boarding_process': 1, 'baggage_issues': 0, 'inflight_experience': 1, 'pricing_fees': 1, 'online_booking': 0}
{'flight_delay_cancellation': -1, 'checkin_boarding_process': 0, 'baggage_issues': 0, 'inflight_experience': 0, 'pricing_fees': 0, 'online_booking': -1}
{'flight_delay_cancellation': 0, 'checkin_boarding_process': -1, 'baggage_issues': 0, 'inflight_experience': 0, 'pricing_fees': -1, 'online_booking': 0}
{'flight_delay_cancellation': -1, 'checkin_boarding_process': 0, 'baggage_issues': 0, 'inflight_experience': 1, 'pricing_fees': 1, 'online_booking': 0}
{'flight_delay_cancellation': 1, 'checkin_boarding_process': 0, 'baggage_issues': 0, 'inflight_experience': 1, 'pricing_fees': 0, 'online_booking': 0}
{'flight_delay_cancellation': 0, 'checkin_boarding_process': 0, 'baggage_issues': -1, 'i

In [55]:
def add_synthetic_dates(df, start_date="2025-11-10", end_date="2025-11-30"):
    """
    Add synthetic dates uniformly distributed across date range.
    
    Args:
        df: DataFrame to add dates to
        start_date: Start date as string (YYYY-MM-DD)
        end_date: End date as string (YYYY-MM-DD)
    
    Returns:
        DataFrame with 'date' column added
    """
    # Convert string dates to datetime
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)
    
    # Generate uniform dates for each row
    num_rows = len(df)
    dates = pd.date_range(start=start, end=end, periods=num_rows)
    
    df['date'] = dates
    return df
df_merged = add_synthetic_dates(df_merged, start_date="2025-11-10", end_date="2025-11-30")

print(df_merged)

                                                 review  \
0     Dont buy the business combo or flexibility or ...   
1     I wasnt quite sure what to expect having read ...   
2     Los Angeles to Copenhagen via Reykjavik. Ive m...   
3     Never had such a poor experience time after ti...   
4     Fabulous flights with AirAsia X yet again. Bot...   
...                                                 ...   
4995  Southwest has had major delays on my past four...   
4996  For a 3.45 hour Helsinki to Athens flight we g...   
4997  My girlfriend and I flew LIAT from into St. Ma...   
4998  Bad business practices on multiple levels. I h...   
4999  I was Traveling to Egypt from Miami via Frankf...   

                                                 labels  \
0                                 pricing_fees_negative   
1     checkin_boarding_process_positive | inflight_e...   
2     flight_delay_cancellation_negative | online_bo...   
3     checkin_boarding_process_negative | pricing_fe...

In [50]:
print(df_merged.loc[:, df_merged.columns != 'review'].iloc[30:40].to_string())

                                                                                      labels  flight_delay_cancellation  checkin_boarding_process  baggage_issues  inflight_experience  pricing_fees  online_booking
30                                                              inflight_experience_positive                          0                         0               0                    1             0               0
31                         flight_delay_cancellation_negative | inflight_experience_positive                         -1                         0               0                    1             0               0
32                                    inflight_experience_positive | baggage_issues_positive                          0                         0               1                    1             0               0
33                    flight_delay_cancellation_negative | checkin_boarding_process_negative                         -1                        -1   

In [51]:
df_merged.dtypes #columns

review                       object
labels                       object
flight_delay_cancellation     int64
checkin_boarding_process      int64
baggage_issues                int64
inflight_experience           int64
pricing_fees                  int64
online_booking                int64
dtype: object

In [56]:
df_merged.to_csv('../data/raw/labeled_data.csv', index = False)

In [53]:
df_merged

,review,labels,flight_delay_cancellation,checkin_boarding_process,baggage_issues,inflight_experience,pricing_fees,online_booking
0,Dont buy the business combo or flexibility or ...,pricing_fees_negative,0,0,0,0,-1,0
1,I wasnt quite sure what to expect having read ...,checkin_boarding_process_positive | inflight_e...,0,1,0,1,1,0
2,Los Angeles to Copenhagen via Reykjavik. Ive m...,flight_delay_cancellation_negative | online_bo...,-1,0,0,0,0,-1
3,Never had such a poor experience time after ti...,checkin_boarding_process_negative | pricing_fe...,0,-1,0,0,-1,0
4,Fabulous flights with AirAsia X yet again. Bot...,flight_delay_cancellation_negative | inflight_...,-1,0,0,1,1,0
...,...,...,...,...,...,...,...,...
4995,Southwest has had major delays on my past four...,flight_delay_cancellation_negative,-1,0,0,0,0,0
4996,For a 3.45 hour Helsinki to Athens flight we g...,flight_delay_cancellation_negative | inflight_...,-1,0,0,-1,0,0
4997,My girlfriend and I flew LIAT from into St. Ma...,flight_delay_cancellation_negative | checkin_b...,-1,-1,0,0,0,0
4998,Bad business practices on multiple levels. I h...,pricing_fees_negative,0,0,0,0,-1,0
